## Week 3 – Order Analysis with PySpark

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, count


In [0]:
# Initialize Spark
spark = SparkSession.builder.appName("Order Analysis").getOrCreate()


In [0]:
# Load data
orders_df = spark.read.csv("/Volumes/workspace/default/subramani/orders (4).csv", header=True, inferSchema=True)
customers_df = spark.read.csv("/Volumes/workspace/default/subramani/customers (3).csv", header=True, inferSchema=True)


In [0]:
# Join on customer ID
joined_df = orders_df.join(customers_df, on="customer_id", how="inner")
joined_df.show(5)

+-----------+--------+----------+-------------+-------------+---------------+-----------------+------+--------------------+------------+
|customer_id|order_id|order_date|expected_date|delivery_date|        product|    customer_name|region|               email|       phone|
+-----------+--------+----------+-------------+-------------+---------------+-----------------+------+--------------------+------------+
|          2|      93|2025-04-22|   2025-05-26|   2025-06-13|             TV|      Arnav Dayal|  West|chandranfateh@sac...|910025040216|
|          3|      33|2025-05-14|   2025-05-31|   2025-06-10|   Refrigerator|Yuvaan Srinivasan| North|divyanshkhanna@ya...|  3968373591|
|          4|      10|2025-05-18|   2025-06-06|   2025-05-29|Washing Machine|        Anvi Gola|  East|devanalisha@venka...|  1558754080|
|          5|      21|2025-04-27|   2025-05-24|   2025-06-03|         Laptop|    Dishani Boase|  West|viswanathanindran...|915138533444|
|          6|      99|2025-04-30|   2025-

In [0]:

# Add delay flag
joined_df = joined_df.withColumn("is_delayed", when(col("delivery_date") < col("expected_date"), 1).otherwise(0))

# Group by region and count delays
delay_by_region = joined_df.groupBy("region").agg(count(when(col("is_delayed") == 1, True)).alias("delay_count"))

# Save output
# Save output to a permitted path
delay_by_region.show()

delay_by_region.write.mode("overwrite").csv("/Volumes/workspace/default/subramani/delays_by_region.csv")


+------+-----------+
|region|delay_count|
+------+-----------+
|  West|          8|
|  East|          9|
| North|          6|
| South|          9|
+------+-----------+

